# getitem-back-add-at — faded example 3: Fill the upstream gradient for the manual check

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`. Running the beacon reports progress on the `Backprop: getitem_back via add-at` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: getitem_back via add-at` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`getitem-back-add-at`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "getitem-back-add-at"
DD_SUBTOPIC = "Backprop: getitem_back via add-at"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To cross-check the scatter-add backward against autograd, the manual side needs the upstream gradient `grad_out = d(loss)/d(out)`. For `loss = (out * w).sum()`, that gradient is exactly `w`.

## Faded exercise 3

A leaf `x` is gathered to `out = x[idx]` and reduced via `loss = (out * w).sum()`; autograd fills `x.grad`. Compute the same gradient manually with `index_add_`. Complete the blanked `grad_out` (the gradient of the loss with respect to `out`).

**Fill in:** the upstream gradient d(loss)/d(out), which equals w for this weighted-sum loss

In [ ]:
import torch as t

t.manual_seed(5)
x = t.randn(4, 3, requires_grad=True)
idx = t.tensor([1, 1, 3])
w = t.randn(3, 3)

out = x[idx]
(out * w).sum().backward()

grad_out = w
manual = t.zeros_like(x)
manual.index_add_(0, idx, grad_out)
print(bool(t.allclose(x.grad, manual)))


def _test():
    # the manual scatter-add gradient must equal autograd's x.grad
    assert t.allclose(x.grad, manual), (x.grad, manual)
    # row 1 is repeated -> it accumulates two rows of w
    expected_row1 = w[0] + w[1]
    assert t.allclose(manual[1], expected_row1, atol=1e-5), (manual[1], expected_row1)
    # row 0 was never indexed -> zero gradient
    assert t.allclose(manual[0], t.zeros(3))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(5)
x = t.randn(4, 3, requires_grad=True)
idx = t.tensor([1, 1, 3])
w = t.randn(3, 3)

out = x[idx]
(out * w).sum().backward()

grad_out = w
manual = t.zeros_like(x)
manual.index_add_(0, idx, grad_out)
print(bool(t.allclose(x.grad, manual)))
```
</details>